# Notebook 4: Beyond the Basics

## What You'll Learn

The first three notebooks taught you the **core** of consistent hashing.
This notebook covers four important real-world extensions:

| # | Topic | Why It Matters |
|---|-------|----------------|
| 1 | **Replication on the ring** | Keep data alive when a node dies (the *real* fault-tolerance trick) |
| 2 | **Weighted nodes** | Big servers should hold more data than small ones |
| 3 | **Rendezvous hashing (HRW)** | A simpler alternative to the ring - when is each better? |
| 4 | **Hot keys** (brief) | What hashing *can't* solve, and what to do about it |

> **Prerequisites:** Notebooks 1-3 (you should be comfortable with the `ConsistentHashRing` idea).


In [ ]:
import hashlib
import bisect
from collections import Counter, defaultdict


---

## Part 1: Replication on the Ring

In Notebook 3 we noticed: if a Redis node dies, the keys that lived **only** on
that node are gone. Real systems fix this with **replication**: every key is
stored on **N nodes**, not just one.

**The classic Dynamo recipe:** write the key to its primary node *and* the next
**N - 1** distinct physical nodes clockwise on the ring. If any one node dies,
the key is still readable from a replica.

```
            +---------+
        S3            S1     <- key K hashes here
                                K -> S1, S2, S4 (RF=3)
        S2            S4
            +---------+
```

> WARNING: **Common bug to avoid.** When you walk clockwise, you must skip vnodes
> whose physical node you've already chosen. Otherwise a "replica" might just be
> the same server again - useless for fault tolerance.


In [ ]:
class ReplicatedHashRing:
    """Consistent hash ring that returns N DISTINCT physical replicas per key."""

    def __init__(self, num_virtual_nodes: int = 150, replication_factor: int = 3):
        self.num_virtual_nodes = num_virtual_nodes
        self.replication_factor = replication_factor
        self.ring = {}             # ring position -> physical node name
        self.sorted_positions = []
        self.nodes = set()

    def _hash(self, key: str) -> int:
        return int(hashlib.md5(key.encode()).hexdigest(), 16) % (2**32)

    def add_node(self, node: str):
        self.nodes.add(node)
        for i in range(self.num_virtual_nodes):
            pos = self._hash(f"{node}#vn{i}")
            self.ring[pos] = node
            bisect.insort(self.sorted_positions, pos)

    def remove_node(self, node: str):
        self.nodes.discard(node)
        for i in range(self.num_virtual_nodes):
            pos = self._hash(f"{node}#vn{i}")
            if pos in self.ring:
                del self.ring[pos]
                self.sorted_positions.remove(pos)

    def get_replicas(self, key: str) -> list:
        """Return up to `replication_factor` DISTINCT physical nodes for this key."""
        if not self.ring:
            return []
        pos = self._hash(key)
        idx = bisect.bisect_right(self.sorted_positions, pos)

        replicas = []
        seen = set()
        # Walk clockwise around the ring at most once
        for step in range(len(self.sorted_positions)):
            i = (idx + step) % len(self.sorted_positions)
            node = self.ring[self.sorted_positions[i]]
            if node not in seen:
                seen.add(node)
                replicas.append(node)
                if len(replicas) == self.replication_factor:
                    break
        return replicas

print("ReplicatedHashRing ready!")


### Demo: store keys on 3 replicas, then "kill" a node


In [ ]:
# Build a 5-node ring with replication factor 3
ring = ReplicatedHashRing(num_virtual_nodes=150, replication_factor=3)
for s in ["S1", "S2", "S3", "S4", "S5"]:
    ring.add_node(s)

# Quick sanity check: every replica list has 3 distinct nodes
sample_key = "user:42"
print(f"Replicas for '{sample_key}': {ring.get_replicas(sample_key)}")

# Simulate "storing" 1,000 keys: each key written to all 3 replicas
storage = defaultdict(set)   # node -> set of keys it holds
for i in range(1000):
    key = f"user:{i}"
    for replica in ring.get_replicas(key):
        storage[replica].add(key)

print("\nKeys stored per node (each key lives on 3 nodes):")
for node in sorted(storage):
    print(f"  {node}: {len(storage[node]):>4} keys")

total_writes = sum(len(v) for v in storage.values())
print(f"\n  Total physical writes: {total_writes:,} (= 1,000 keys * RF=3)")


In [ ]:
# Now S3 crashes. How many keys can we still read?
dead_node = "S3"
readable = 0
unreadable = 0
for i in range(1000):
    key = f"user:{i}"
    replicas = ring.get_replicas(key)
    # A key is still readable if at least one of its replicas is alive
    alive_replicas = [r for r in replicas if r != dead_node]
    if alive_replicas:
        readable += 1
    else:
        unreadable += 1

print(f"{dead_node} just died.")
print(f"   Readable keys:   {readable:,} / 1,000  (replicas saved us)")
print(f"   Unreadable keys: {unreadable:,} / 1,000")
print()
print(f"   With RF=3, ALL keys survive a single-node failure.")
print(f"   You'd need to lose 3 SPECIFIC nodes simultaneously to lose any data.")


**Note the separation of concerns:**

- The **hash ring** decides *where* keys go.
- **Replication** decides *how many copies* to keep.
- **Membership changes** (adding/removing nodes from the ring) is yet another step.

Mixing these up is the #1 source of confusion in distributed systems interviews.

---

## Part 2: Weighted Nodes

Real clusters aren't homogeneous. You might have:

| Node | Capacity |
|------|----------|
| S-small  | 1 unit (e.g. 16 GB RAM) |
| S-medium | 2 units (32 GB) |
| S-large  | 4 units (64 GB) |

A naive ring would give each server roughly the same number of keys - overloading
the small one. The fix is simple: **give bigger nodes more virtual nodes.**

> TIP: This is *approximate* (vnode hashing is random), so the more vnodes you use,
> the closer you get to the target weights.


In [ ]:
class WeightedHashRing:
    """Consistent hash ring where each node can have a custom weight."""

    def __init__(self, vnodes_per_unit: int = 100):
        # vnodes for a node = weight * vnodes_per_unit
        self.vnodes_per_unit = vnodes_per_unit
        self.ring = {}
        self.sorted_positions = []
        self.weights = {}   # node -> weight

    def _hash(self, key: str) -> int:
        return int(hashlib.md5(key.encode()).hexdigest(), 16) % (2**32)

    def add_node(self, node: str, weight: int = 1):
        self.weights[node] = weight
        for i in range(weight * self.vnodes_per_unit):
            pos = self._hash(f"{node}#vn{i}")
            self.ring[pos] = node
            bisect.insort(self.sorted_positions, pos)

    def get_node(self, key: str) -> str:
        if not self.ring:
            return None
        pos = self._hash(key)
        idx = bisect.bisect_right(self.sorted_positions, pos)
        if idx == len(self.sorted_positions):
            idx = 0
        return self.ring[self.sorted_positions[idx]]


# Build a weighted ring: small=1x, medium=2x, large=4x
wring = WeightedHashRing(vnodes_per_unit=200)
wring.add_node("S-small",  weight=1)
wring.add_node("S-medium", weight=2)
wring.add_node("S-large",  weight=4)

# Distribute 70,000 keys
NUM_KEYS = 70_000
counts = Counter(wring.get_node(f"key:{i}") for i in range(NUM_KEYS))

print("Distribution with weights 1 / 2 / 4 (target ratio):\n")
total_weight = sum(wring.weights.values())   # 7
for node, weight in sorted(wring.weights.items()):
    actual = counts[node]
    pct_actual = actual / NUM_KEYS * 100
    pct_expected = weight / total_weight * 100
    bar = "#" * (actual // 1000)
    print(f"  {node:<10} weight={weight}  actual={actual:>6,} ({pct_actual:5.1f}%)  "
          f"target={pct_expected:5.1f}%  {bar}")


The actual percentages closely track the targets (~14% / 29% / 57%).
With fewer vnodes-per-unit the distribution gets noisier - try changing
`vnodes_per_unit=200` to `20` and re-running to see this.

---

## Part 3: Rendezvous Hashing (HRW)

**Rendezvous hashing** (also called **Highest Random Weight, HRW**) is a different
way to map keys to nodes. There's no ring at all - instead:

> For each key, compute `hash(key, node)` for **every node** and pick the node
> with the **highest hash value**.

```
key "user:42"
    |- score(key, S1) = 0.81
    |- score(key, S2) = 0.93   <- winner
    |- score(key, S3) = 0.42
    |- score(key, S4) = 0.17
```

That's the entire algorithm. No vnodes, no sorted positions, no binary search.


In [ ]:
def hrw_hash(key: str, node: str) -> int:
    """Score a (key, node) pair. Higher score = better choice."""
    return int(hashlib.md5(f"{key}|{node}".encode()).hexdigest(), 16)


class RendezvousHashing:
    """Highest Random Weight (HRW) hashing - no ring needed."""

    def __init__(self):
        self.nodes = set()

    def add_node(self, node: str):
        self.nodes.add(node)

    def remove_node(self, node: str):
        self.nodes.discard(node)

    def get_node(self, key: str) -> str:
        if not self.nodes:
            return None
        # Pick the node with the highest score for this key
        return max(self.nodes, key=lambda n: hrw_hash(key, n))

    def get_top_n(self, key: str, n: int) -> list:
        """Return the top-N scoring nodes (e.g. for replication)."""
        return sorted(self.nodes, key=lambda n: hrw_hash(key, n), reverse=True)[:n]


print("RendezvousHashing ready!")


In [ ]:
# Try it out
hrw = RendezvousHashing()
for s in ["S1", "S2", "S3", "S4"]:
    hrw.add_node(s)

for key in ["user:1", "user:2", "user:3", "user:42"]:
    print(f"  '{key}' -> {hrw.get_node(key)}   "
          f"(top 3 for replication: {hrw.get_top_n(key, 3)})")


### Fair comparison: HRW vs Consistent Hash Ring

To be fair, we'll compare HRW to a ring with **150 vnodes per server**
(Notebook 2's "sweet spot"), using the **same** 10,000 keys and **same** 4 nodes.
We measure two things that matter in production:

1. **Balance** - are keys spread evenly?
2. **Stability** - when we add a 5th node, what fraction of keys must move?


In [ ]:
# Compact ring class (same algorithm as Notebook 2)
class SimpleRing:
    def __init__(self, vnodes=150):
        self.vnodes = vnodes
        self.ring = {}
        self.sp = []
        self.nodes = set()
    def _h(self, k):
        return int(hashlib.md5(k.encode()).hexdigest(), 16)
    def add(self, n):
        self.nodes.add(n)
        for i in range(self.vnodes):
            p = self._h(f"{n}#vn{i}")
            self.ring[p] = n
            bisect.insort(self.sp, p)
    def get(self, k):
        if not self.sp:
            return None
        p = self._h(k)
        i = bisect.bisect_right(self.sp, p)
        if i == len(self.sp):
            i = 0
        return self.ring[self.sp[i]]


NUM_KEYS = 10_000
keys = [f"key:{i}" for i in range(NUM_KEYS)]
nodes_4 = ["S1", "S2", "S3", "S4"]

ring = SimpleRing(vnodes=150)
hrw  = RendezvousHashing()
for n in nodes_4:
    ring.add(n)
    hrw.add_node(n)

ring_dist = Counter(ring.get(k) for k in keys)
hrw_dist  = Counter(hrw.get_node(k) for k in keys)


def stats(dist, label):
    vals = list(dist.values())
    ideal = NUM_KEYS / len(vals)
    sd = (sum((v - ideal) ** 2 for v in vals) / len(vals)) ** 0.5
    print(f"  {label:<25} max/min={max(vals)/min(vals):.3f}  std_dev={sd:>6.1f}")


print("Balance (4 nodes, 10,000 keys):")
stats(ring_dist, "Ring (150 vnodes)")
stats(hrw_dist,  "Rendezvous (HRW)")


In [ ]:
# Stability: snapshot, then add a 5th node, count how many keys moved.
ring_before = {k: ring.get(k)     for k in keys}
hrw_before  = {k: hrw.get_node(k) for k in keys}

ring.add("S5")
hrw.add_node("S5")

ring_moved = sum(1 for k in keys if ring_before[k] != ring.get(k))
hrw_moved  = sum(1 for k in keys if hrw_before[k]  != hrw.get_node(k))

ideal_pct = 100 / 5  # ~20% should move when going 4 -> 5 nodes

print(f"Stability when adding a 5th node (ideal ~ {ideal_pct:.0f}% moved):")
print(f"  Ring (150 vnodes):  {ring_moved:>5,} keys moved  ({ring_moved/NUM_KEYS*100:.1f}%)")
print(f"  Rendezvous (HRW):   {hrw_moved:>5,} keys moved  ({hrw_moved/NUM_KEYS*100:.1f}%)")
print()
print("Both algorithms are stable: only ~1/N of keys move when adding a node.")


### When to choose which?

| | Consistent Hash Ring | Rendezvous Hashing (HRW) |
|---|---|---|
| **Lookup cost** | O(log N) (binary search over vnodes) | O(N) (score against every node) |
| **Memory** | O(N x vnodes) - many positions stored | O(N) - just the node list |
| **Balance** | Needs ~150 vnodes/node to be even | Naturally even, no tuning |
| **Replication** | "Next N clockwise (distinct)" | "Top N by score" - even simpler |
| **Best for** | Large clusters (100s-1000s of nodes) | Small / medium clusters, simplicity |

**Real-world users of HRW:** Apache Druid, GitHub's load balancer (GLB), Akamai's
[CARP](https://en.wikipedia.org/wiki/Cache_Array_Routing_Protocol), and many CDN
edge selectors. It's a great default when you have <= ~100 nodes.

---

## Part 4: Hot Keys (brief)

WARNING: Neither consistent hashing **nor** rendezvous hashing solves the
**hot key** problem. If one key (say `user:beyonce`) is requested 1 million times
per second, it lives on exactly **one** primary node - and that node will melt.

Hashing is about **placement**, not **load**. Common fixes layered *on top* of
hashing:

| Technique | Idea |
|-----------|------|
| **Read replicas** | Multiple nodes serve reads for the same key (eventual consistency) |
| **Key splitting** | `user:beyonce#1` ... `user:beyonce#10` spread across nodes; client picks one at random |
| **Local caching** | Cache hot keys in the app process (e.g. with TTL) |
| **Bounded-load consistent hashing** | Cap each node at `(1 + epsilon) * avg`; spill overflow to the next node (used at Google / Vimeo) |
| **Power of two choices** | Hash to two candidates, pick the one with lower current load |

> **Interview gotcha:** if the question is "you have one extremely popular item",
> the answer is **not** "use consistent hashing" - it's caching, replication, or
> key splitting. Hashing alone won't help.

---

## Key Takeaways

1. **Replication on the ring** = walk clockwise, but skip vnodes whose physical
   node you've already chosen. RF=3 is the industry default (DynamoDB, Cassandra).
2. **Weighted nodes** = scale `num_virtual_nodes` by the node's capacity. Use
   plenty of vnodes-per-unit so the approximation is tight.
3. **Rendezvous hashing** is a beautifully simple alternative - pick the node
   with the highest `hash(key, node)`. Great for <= ~100 nodes.
4. **Hashing doesn't solve hot keys.** That's a *load* problem, not a *placement*
   problem.

## Further Reading

- [Consistent Hashing with Bounded Loads (Google, 2017)](https://arxiv.org/abs/1608.01350)
- [Rendezvous Hashing (Wikipedia)](https://en.wikipedia.org/wiki/Rendezvous_hashing)
- [Jump Consistent Hash (Google, 2014)](https://arxiv.org/abs/1406.2294) - another modern alternative
- The Dynamo paper (linked in Notebook 3) for replication on the ring
